In [ ]:
import ee
import os

# --- Earth Engine Initialization ---
try:
    # Ensure you have authenticated with 'earthengine authenticate'
    ee.Initialize(project="gsapp-map")
    print("Earth Engine Initialized.")
except Exception as e:
    print(f"Error initializing Earth Engine: {e}")
    # Exit or handle as needed for your environment

# --- USER CONFIGURATION ---
# IMPORTANT: The 'samples' FeatureCollection (reference locations) must be defined.
try:
    # --- PLACEHOLDER: Define or Load the 'samples' FeatureCollection ---
    # Define a single point near a silo location in Franklin County, KS
    samples = ee.FeatureCollection(
        [
            ee.Feature(ee.Geometry.Point([-95.20, 38.60])),
        ],
        ["name"],
    )
    print("Reference 'samples' loaded/defined.")
except Exception as e:
    print(f"Could not load samples: {e}. Please define the 'samples' variable.")
    exit()

# Set the target year
year = 2024
startDate = ee.Date.fromYMD(year, 1, 1)
endDate = startDate.advance(1, "year")

# --- Step 0: Select the Search Region (Franklin County, Kansas) ---
print("Selecting search region...")
counties = ee.FeatureCollection("TIGER/2018/Counties")
selected_county = counties.filter(ee.Filter.eq("GEOID", "20059"))
geometry = selected_county.geometry()

# --- Step 1: Filter and Mosaic the Satellite Embedding Dataset ---
print(f"Loading embeddings for {year}...")
embeddings = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
mosaic = embeddings.filterDate(startDate, endDate).mosaic()
bandNames = mosaic.bandNames()

# --- Step 2: Extract the Embedding Vector from the Samples ---
scale = 100  # Choose the scale for sampling and vectorization (meters)
print(f"Sampling reference embeddings at scale: {scale}m...")
sampleEmbeddings = mosaic.sampleRegions(collection=samples, scale=scale)

# --- Step 3: Calculate Similarity (Dot Product) ---
print("Calculating similarity (mean dot product)...")


def calculate_dot_product(f):
    """Calculates the dot product between the sample embedding and the mosaic."""
    arrayImage = ee.Image(f.toArray(bandNames)).arrayFlatten([bandNames])
    dotProduct = arrayImage.multiply(mosaic).reduce("sum").rename("similarity")
    return dotProduct


sampleDistances = ee.ImageCollection(sampleEmbeddings.map(calculate_dot_product))
meanDistance = sampleDistances.mean()

# --- Step 4: Visualize the Distance Image (Prepared) ---
palette = [
    "000004",
    "2C105C",
    "711F81",
    "B63679",
    "EE605E",
    "FDAE78",
    "FCFDBF",
    "FFFFFF",
]
similarityVis = {"palette": palette, "min": -1, "max": 1}
# The 'meanDistance' image is now ready for visualization.

# --- Step 5: Extract Location Matches ---
threshold = 0.90  # Apply a threshold (higher = stricter match)
print(f"Applying threshold ({threshold}) and vectorizing matches...")
similarPixels = meanDistance.gt(threshold)

# Vectorize the results
# Mask 0 values using selfMask() to get polygons only for the matched pixels
# NOTE ON MAXPIXELS: This must be limited for client-side processing.
# 1e6 is 1 million pixels, which is usually the hard limit for export/client operations.
# Create a small area (e.g., a 10km buffer around a sample point)
small_area = samples.geometry().buffer(5000).bounds()

polygons = similarPixels.selfMask().reduceToVectors(
    scale=scale,
    eightConnected=False,
    maxPixels=1e6,
    # CHANGE: Use the smaller geometry for clipping
    geometry=small_area,  # <--- Use the smaller geometry here
)
# ... rest of the code ...

# Extract the centroids of vectorized polygons
predictedMatches = polygons.map(lambda f: f.centroid(maxError=1))

# --- Step 6: Bring Results to Client (NO EXPORT) ---
# To use the results immediately, we must call .getInfo() on the FeatureCollection.
# WARNING: If the result is too large (more than 5000 features or 1MB), this will fail.

print("\n--- WARNING: Fetching results to client (getInfo()). ---")
print("This may fail if the output FeatureCollection is too large.")

try:
    final_matches_client = predictedMatches.getInfo()
    print(
        f"\nSuccessfully fetched {len(final_matches_client['features'])} predicted matches to client memory."
    )

    # The 'final_matches_client' is now a standard Python dictionary (GeoJSON format).
    # You can process or visualize it immediately using libraries like folium or geopandas.

except Exception as e:
    print(f"\nERROR: Failed to fetch results to client memory (getInfo()).")
    print("The most likely reason is the result size exceeds the memory limits.")
    print("If this happens, you MUST use the Export.table.toAsset method.")


# --- Visualization Snippet (For Jupyter/Colab) ---
# To actually see the results, you need a visualization setup (like folium/ee.mapclient)
# If you were in a Colab notebook, you could display the results like this:

import folium
import ee.mapclient

Map = folium.Map(location=[38.65, -95.25], zoom_start=10)
Map.add_ee_layer(geometry, {"color": "red", "opacity": 0.5}, "Search Area")
Map.add_ee_layer(
    meanDistance.clip(geometry), similarityVis, "Similarity (bright = close)", False
)
Map.add_ee_layer(predictedMatches, {"color": "cyan"}, "Predicted Matches")
display(Map)

Earth Engine Initialized.
Reference 'samples' loaded/defined.
Selecting search region...
Loading embeddings for 2024...
Sampling reference embeddings at scale: 100m...
Calculating similarity (mean dot product)...
Applying threshold (0.9) and vectorizing matches...

--- WARNING: Fetching results to client (getInfo()). ---
This may fail if the output FeatureCollection is too large.

Successfully fetched 15 predicted matches to client memory.


AttributeError: 'Map' object has no attribute 'add_layer'

In [23]:
import ee
import leafmap.maplibregl as leafmap

ee.Authenticate()
ee.Initialize(project="gsapp-map")

m = leafmap.Map(
    projection="globe", sidebar_visible=True, center=[-100, 40], zoom=2, height="800px"
)
google_hybrid = "https://mt1.google.com/vt/lyrs=y&hl=en&x={x}&y={y}&z={z}"
m.add_tile_layer(google_hybrid, name="Google Hybrid", attribution="Google")
m.add_similarity_search(
    default_year=2024,
    default_color="#0000ff",
    default_threshold=0.99,
)
m.add_draw_control(controls=["point", "trash"])
m

Html(children=[Map(calls=[['addControl', ('NavigationControl', {'showCompass': True, 'showZoom': True, 'visual…

In [ ]:
import geemap
import time
import math

# --- Configuration ---

# 1. Define your Query Location (Example: A structure in the Texas Panhandle)
QUERY_LAT = 35.1952
QUERY_LON = -101.8458
query_point = ee.Geometry.Point(QUERY_LON, QUERY_LAT)

# 2. Define the Search Area (Setting the geometry to the global land area)
# Using the bounds of a large reliable feature collection (TIGER/2018/Country).
# Note: We must buffer the bounds to ensure coverage across all tiles.
us_boundary = ee.FeatureCollection("TIGER/2018/Country").first()
# Using a 10,000 km radius circle as a proxy for 'global land area' for speed
search_area = query_point.buffer(10000000).bounds()

# 3. Define the Search Resolution (Scale)
# WARNING: Setting this too low (e.g., 10m) for a global search will likely crash.
SEARCH_SCALE = 1000  # meters (Higher is faster but less precise)


def find_most_similar_location(query_pt, search_geom, scale):
    """
    Finds the pixel with the highest embedding similarity score within the search geometry.
    """

    # --- 1. Load and Process Embeddings ---
    print(f"\nSearching for match using scale: {scale}m...")
    year = 2024
    startDate = ee.Date.fromYMD(year, 1, 1)
    endDate = startDate.advance(1, "year")

    embeddings = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    mosaic = embeddings.filterDate(startDate, endDate).mosaic()
    bandNames = mosaic.bandNames()

    # --- 2. Extract Query Embedding Vector (FIXED FOR .toArray() ERROR) ---
    query_collection = mosaic.sampleRegions(
        collection=ee.FeatureCollection(query_pt), scale=scale, tileScale=16
    )

    query_feature = ee.Feature(query_collection.first())

    # Convert the feature properties into a single-pixel image vector
    query_vector_image = ee.Image(query_feature.toArray(bandNames)).arrayFlatten(
        [bandNames]
    )

    # --- 3. Calculate Similarity Image (Dot Product) ---
    # Similarity = sum(mosaic * query_vector_image)
    similarity_image = (
        query_vector_image.multiply(mosaic).reduce("sum").rename("similarity")
    )

    # --- 4. Prepare for Reduction (FIXED FOR Reducer.max ERROR) ---
    # Add coordinate bands so the reducer can return location (X, Y)
    coords = ee.Image.pixelLonLat()

    # Resulting image has bands: ['similarity', 'longitude', 'latitude']
    image_with_coords = similarity_image.addBands(coords)

    # --- 5. Find the Maximum Value and Location ---
    print(
        "Starting global maximum similarity reduction (This may take several minutes)..."
    )
    start_time = time.time()

    max_result = image_with_coords.reduceRegion(
        reducer=ee.Reducer.max(
            3
        ),  # Get max of the 3 bands: similarity, longitude, latitude
        geometry=search_geom,
        scale=scale,
        maxPixels=1e13,  # Extremely high limit for server-side processing
        bestEffort=True,
        tileScale=16,
    )

    result_data = max_result.getInfo()

    end_time = time.time()
    print(f"Reduction complete in {end_time - start_time:.2f} seconds.")

    # --- 6. Extract and Format Results ---
    # The reducer returns keys corresponding to the max value of each band.
    closest_dist = result_data.get("similarity")
    closest_lon = result_data.get("longitude")
    closest_lat = result_data.get("latitude")

    if closest_dist is None or closest_lon is None:
        print("\nERROR: No result found. Try increasing SEARCH_SCALE.")
        return similarity_image, None, None

    closest_pt = ee.Geometry.Point(closest_lon, closest_lat)

    return similarity_image, closest_dist, closest_pt


# --- Execution ---
similarity_image, closest_dist, closest_pt = find_most_similar_location(
    query_point, search_area, SEARCH_SCALE
)


# --- Output and Visualization ---

if closest_pt:
    closest_lon, closest_lat = closest_pt.coordinates().getInfo()

    print("\n--- RESULTS ---")
    print(f"Query Location: ({QUERY_LAT:.4f}, {QUERY_LON:.4f})")
    print(f"Closest Location: ({closest_lat:.4f}, {closest_lon:.4f})")
    print(f"Highest Similarity Score: {closest_dist:.6f} (Max possible is 1.0)")

    # --- Visualization ---
    Map = geemap.Map(center=[QUERY_LAT, QUERY_LON], zoom=5)
    Map.add_basemap("SATELLITE")

    # Similarity Image Visualization
    similarityVis = {
        "palette": [
            "000004",
            "2C105C",
            "711F81",
            "B63679",
            "EE605E",
            "FDAE78",
            "FCFDBF",
            "FFFFFF",
        ],
        "min": math.floor(
            closest_dist - 0.05
        ),  # Set min/max based on results for better contrast
        "max": 1.0,
    }

    # Add Similarity Layer (Clipped to a manageable area for visualization performance)
    Map.addLayer(
        similarity_image.clip(
            query_point.buffer(50000)
        ),  # Clip to 50km around the query for local detail
        similarityVis,
        f"Similarity Map ({SEARCH_SCALE}m)",
        True,
    )

    # Query Point
    Map.addLayer(query_point, {"color": "red"}, "Query Location")

    # Closest Match Point
    Map.addLayer(
        closest_pt, {"color": "cyan", "pointRadius": 8}, "Closest Match Location"
    )

    # Center map to show both points (or zoom out for context)
    Map.centerObject(query_point.centroid(1), 8)

    Map


Searching for match using scale: 1000m...
Starting global maximum similarity reduction (This may take several minutes)...


EEException: Computation timed out.